# BilbyFlow Quick Start

You have reached the BilbyFLow quick start. This will walk through the in-built CLI methods that will allow you to very quickly start training/evaluating your own Gravitational Wave (GW) Neural Posterior Estimator(NPE). Rest of the tutorials deal with design philosophy and how to customize different aspects of the pipeline.

## Installing BilbyFlow

As the code currently stands, it is pip installable once you have cloned the code from GitHub. Very soon it will be uploaded to PyPI, but for now the steps are below.

#### 1. Clone the project from GitHub

Preusming you have git already installed (if not then head [here](https://git-scm.com/install/)) all one needs to do to get the `BilbyFlow` code from source is

```bash
git clone https://github.com/LiamCPinchbeck/BilbyFlow.git
```

And then the code should be in your working directory.

#### 2. Create a Python Venv

With python installed, you can create a virtual env with the following via pip

```bash
python -m venv .VENVNAME
```

#### 3. Install

Once the code is in your working directory, and you've activated your favourite python environment, then it should just be a local pip-install (workign directory has to be the same as that of the code, otherwise it's a relative/absolute path to it)

```bash
pip install .
```

## Copying BilbyFLow CLI scripts to working directory as examples

Once `BilbyFlow` is installed, you can download the `scripts` folder of the code to your local directory.

```bash
python -m bilbyflow.scripts.init_scripts
```

These can then be used from command line like with the cli, or as examples. Some options include the below.

```bash
    bilbyflow-init                        # copies to ./
    bilbyflow-init /path/to/my/run        # copies there
    bilbyflow-init --list                 # just show what would be copied
```

or 

```bash
    bilbyflow.scripts.init_scripts                        # copies to ./
    bilbyflow.scripts.init_scripts /path/to/my/run        # copies there
    bilbyflow.scripts.init_scripts --list                 # just show what would be copied
```


The scripts are copies, not symlinks — edit them freely. The command should also download an example config that runs on a P100 GPU (memory constrained, if you have an A100 or larger you can increase the `micro_batch_size` kwarg in the config).

They import from the installed package, so `pip install -e .` keeps them current with your code changes; only the orchestration (which banks to build, what to plot, CLI flags) lives in the copy.

## Training CLI

With the above installed you should also have an example config in your working directory named `run_config_P100`. The config runs on a __P100__ GPU (up to modifications above). 

With the config or one downloaded from the config example directory in the source code, you can immediately start training an NPE model from the command line.

For the config above you can simply run the code below.

```bash
python -m bilbyflow.scripts.train run_config_P100.yaml
```

And there you go, you have a model being trained. The code should automatically detect a GPU if you have one available and it is compatible with the PyTorch version. From experience, the latest PyTorch version is NOT compatiable with P100 GPUs, but `2.6.0+cu124` is, so you can pip install that if need be. 

This should start training a model with the same configuration as that reported in our paper bar the batch size, number of epoch steps and dataset sizes which would otherwise cause the training to be VERY LONG. 

If you have access to an A100, the comments in the config should allow you to closely reproduce the results up to seed / luck of the draw when it comes to training ML models. We would recommend going through the rest of the tutorials before trying to reproduce results / doing anything custom (which is essentially the whole purpose of the tutorials).

## Efficiency Evaluation CLI

Once you have a trained model, you can evaluate its reweighting efficiency on synthetic injections and/or real GWOSC events.

### Synthetic Injections

Generate injections using bilby (no BilbyFlow machinery, just bilby waveforms + noise), then reweight them with the trained model. This is the controlled test: you know the true parameters, so you can check both efficiency AND correctness.

```bash
# generate 10 synthetic events with network SNR between 8 and 25, min SNR is REQUIRED for comparison to real events, 
    # otherwise there's a pile-up at low SNR due to the powerlaw prior on luminosity distances for the standard Bilby prior
python -m bilbyflow.scripts.make_injections /path/to/trained_model/ \
    --data-dir synth_data --noise-data-dir synth_noise \
    --psd-bank-path /path/to/trained_model/psd_bank.pkl \
    --n-events 10 --min-snr 8 --max-snr 25 --seed 0 --run-tag test1
```

This writes `synth_data/INJ_test1_000_H1.npy` etc. and an event list at `synth_data/events_test1.txt`.

### Reweighting

The reweighting script is the same for both synthetic and real events -- the injections are written in the real-data format on purpose.

```bash
python -m bilbyflow.scripts.reweight_real /path/to/trained_model/ \
    --data-dir synth_data --noise-data-dir synth_noise \
    --events $(cat synth_data/events_test1.txt) \
    --use-checkpoint \
    --checkpoint-path /path/to/trained_model/checkpoint.pt \
    --standardiser-path /path/to/trained_model/standardiser.pkl \
    --n-samples 10000 --npool 8 \
    --waveform-approximant IMRPhenomXPHM \
    --prior-swap --do-single-stage
```

Key flags:
- `--n-samples`: how many posterior draws to evaluate (more = better ESS, slower)
- `--npool`: parallel likelihood evaluations (set to your core count, and make sure `OMP_NUM_THREADS=1` is exported or each worker grabs every core)
- `--prior-swap`: SIR-corrects the flow proposal from the training prior to the physical prior before likelihood evaluation. Almost always want this on.
- `--do-single-stage`: runs the full higher-mode likelihood on all samples directly. The alternative (two-stage) screens with the (2,2) mode first, which is faster but injects variance.

#### Synthetic Reweighting


For synthetic event, the above two steps can be combined and done in a single call

```bash
python -m bilbyflow.scripts.reweight_injections "$MODEL" \
    --psd-bank-path "$MODEL/psd_bank.pkl" \
    --n-events 5 \
    --npool 8 --n-samples 5000 \
    --vary-reference-time \
    --use-checkpoint \
    --checkpoint-path "$MODEL/checkpoint.pt" \
    --standardiser-path "$MODEL/standardiser.pkl" \
    --no-assert \
    --waveform-approximant IMRPhenomXPHM \
    --dt-res 0.00005 \
    --n-phase-basis 5 \
    --positive-harmonics \
    --refine-fac 6 \
    --prior-swap \
    --do-single-stage
```


### Reading the output

Per-event results land in a subdirectory of the model folder:

```
reweight_real_data_synthext_ts_9_priorswap5/
INJ_test1_000_data.pkl # full result dict (samples, weights, diagnostics)
INJ_test1_000_reweighted.png # corner plot
INJ_test1_000_weight_diagnostics.png
summary.txt # one-line-per-event efficiency table
summary.png
```


The numbers to look at:
- **n_eff**: effective sample size after reweighting
- **efficiency**: n_eff / n_valid as a percentage
- **k-hat (PSIS diagnostic)**: below 0.7 is good, 0.7-1.0 is marginal, above 1.0 means the importance weights have a heavy tail and the results may not be reliable. Try more samples or `--prior-swap` with higher `--prior-oversample`.

## Plotting CLI

The reweighting script generates per-event plots automatically, but you can also produce standalone efficiency bar charts from a completed reweighting run.

### Injection efficiency plots

```bash
# from a single reweighting run directory
python -m bilbyflow.scripts.plot_injections /path/to/reweight_output/

# from a batch parent (combines all run_*/summary.pkl found recursively)
python -m bilbyflow.scripts.plot_injections /path/to/batch_parent/

# filter to loud events only
python -m bilbyflow.scripts.plot_injections /path/to/reweight_output/ --dl-max 2000
```

This produces sorted efficiency bar charts (linear and log scale) and an SNR-vs-efficiency panel, which is the diagnostic that tells you whether the model's performance degrades gracefully with distance or falls off a cliff.

### PSIS reliability report

```bash
python -m bilbyflow.scripts.diagnostics /path/to/trained_model/ \
    --reweight-dir /path/to/reweight_output/
```

This writes a `psis_reliability.txt` file summarising the per-event k-hat values and flagging any events where the importance sampling may not be trustworthy. Although funnily enough the k-hat value is not reliable for low-effective sample numbers.

## Downloading `gwosc` Data

Two download scripts handle the data the pipeline needs from GWOSC. Both need internet access (login node on a cluster) and the optional `gwpy` + `gwosc` dependencies (`pip install gwpy gwosc`).

### Off-source noise segments (for the PSD bank)

The training PSD bank is built from real detector noise. Download segments per observing era:

```bash
# see what's available first (no downloads)
python -m bilbyflow.scripts.download_noise --config config.yaml \
    --eras O3a --dry-run

# grab 5 segment pairs per era (~20 min, ~1.3 GB)
python -m bilbyflow.scripts.download_noise --config config.yaml \
    --eras O1 O2 O3a O3b --n-per-era 5
```

`--config` reads `sampling_frequency` and `noise_data_dir` from your training config so the files can't mismatch the model. Without it you must pass `--sr` and `--noise-outdir` explicitly, and a wrong sample rate means the PSD bank builder silently drops every file.

Leave `--n-per-era` unset to tile every available science-mode window (a LOT of data -- always `--dry-run` first). The segments are drawn from the intersection of H1 and L1 data-quality flags, and anything near a catalogued event is vetoed so real signals don't end up in the "noise".

### On-source event strain (for reweighting real events, not implemented yet)

(not implemented yet)

```bash
# list in-prior GWTC events without downloading
python -m bilbyflow.scripts.download_events --config config.yaml \
    --from-gwtc --list-only

# download strain + noise for everything above SNR 12
python -m bilbyflow.scripts.download_events --config config.yaml \
    --from-gwtc --min-snr 12

# or just specific events
python -m bilbyflow.scripts.download_events --config config.yaml \
    --events GW150914 GW170814
```

This writes both the on-source strain (`{outdir}/{event}_{det}.npy`) and the off-source noise file (`{noise_outdir}/{era}/{event}_{det}_noise.npy`) that the reweighting script needs for PSD estimation. Without the noise file, an event cannot be whitened.

The `--from-gwtc` selection keeps events whose published median chirp mass and luminosity distance sit inside the training prior (shrunk inward by `--edge-margin` so the posterior bulk is likely in-prior too), restricted to O1-O3b catalogs by default since the PSD conditioning is trained on those eras.